# 08. Query-Based Summarization

## Goal

Generate a meeting-minute style answer from transcript chunks retrieved for a specific question.

## Why This Step Matters

This is query-based summarization: answer only the user's question using only relevant transcript context, instead of summarizing the whole meeting.

## Inputs

- Cleaned timestamped transcript rows.
- Searchable transcript retrieval from Phase 8.
- Summarization helpers from `src/summarization/`.

## Outputs

- Retrieved transcript context.
- A prompt format for future model-based summarization.
- A conservative extractive meeting-minute answer.

## Setup

Use a small transcript sample first so faithfulness is easy to inspect.

In [1]:
import json
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.retrieval.chunking import chunk_transcript_rows
from src.retrieval.search import search_chunks
from src.summarization.prompts import build_query_summary_prompt
from src.summarization.summarize import summarize_from_context
from src.transcript.clean import clean_transcript_rows

SAVE_OUTPUTS = False
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "summarization"
MAX_CHUNK_CHARACTERS = 70
TOP_K = 1

print("Project root:", PROJECT_ROOT)

Project root: /home/lataeq/Code/Saranote


## Load Data

Start from cleaned transcript rows that contain a meeting decision, an action item, and an unrelated risk note.

In [2]:
raw_rows = [
    {"timestamp": "00:00:00 - 00:00:08", "text": "เปิดประชุม และแจ้งวาระวันนี้"},
    {"timestamp": "00:00:08 - 00:00:18", "text": "ที่ประชุม มีมติ ให้ปรับแผน งบประมาณ สำหรับโครงการใหม่"},
    {"timestamp": "00:00:18 - 00:00:28", "text": "ฝ่ายดำเนินงานต้องส่งเอกสารภายในวันศุกร์"},
    {"timestamp": "00:00:28 - 00:00:38", "text": "มีการพูดถึง ความเสี่ยง เรื่องข้อมูลไม่ครบ"},
]

cleaned_rows = clean_transcript_rows(raw_rows)
chunks = chunk_transcript_rows(cleaned_rows, max_characters=MAX_CHUNK_CHARACTERS)
pd.DataFrame(cleaned_rows)

,timestamp,text
0,00:00:00 - 00:00:08,เปิดประชุม และแจ้งวาระวันนี้
1,00:00:08 - 00:00:18,ที่ประชุม มีมติ ให้ปรับแผน งบประมาณ สำหรับโครง...
2,00:00:18 - 00:00:28,ฝ่ายดำเนินงานต้องส่งเอกสารภายในวันศุกร์
3,00:00:28 - 00:00:38,มีการพูดถึง ความเสี่ยง เรื่องข้อมูลไม่ครบ


## Run Experiment

Retrieve only the transcript chunks related to the question.

In [3]:
question = "ที่ประชุมมีมติเรื่องงบประมาณอย่างไร"
retrieval_results = search_chunks(question, chunks, top_k=TOP_K)
retrieved_context = [result.chunk.text for result in retrieval_results]

context_df = pd.DataFrame([
    {
        "rank": rank,
        "score": round(result.score, 3),
        "start_timestamp": result.chunk.start_timestamp,
        "end_timestamp": result.chunk.end_timestamp,
        "text": result.chunk.text,
    }
    for rank, result in enumerate(retrieval_results, start=1)
])

print("Question:", question)
context_df

Question: ที่ประชุมมีมติเรื่องงบประมาณอย่างไร


,rank,score,start_timestamp,end_timestamp,text
0,1,0.531,00:00:08 - 00:00:18,00:00:08 - 00:00:18,ที่ประชุม มีมติ ให้ปรับแผน งบประมาณ สำหรับโครง...


## Build Prompt

This prompt format is ready for a future open-weight model, but this notebook first uses a conservative extractive answer.

In [4]:
prompt = build_query_summary_prompt(question, "\n".join(retrieved_context))
print(prompt)

Question:
ที่ประชุมมีมติเรื่องงบประมาณอย่างไร

Relevant transcript:
ที่ประชุม มีมติ ให้ปรับแผน งบประมาณ สำหรับโครงการใหม่

Instruction:
สรุปคำตอบจากเนื้อหาการประชุมเป็นภาษาไทย โดยห้ามแต่งข้อมูลที่ไม่มีใน transcript


## Generate Answer

Use only the retrieved transcript lines. If there is no evidence, return a no-evidence message.

In [5]:
answer = summarize_from_context(question, retrieved_context)
print(answer)

จาก transcript ที่เกี่ยวข้อง: ที่ประชุม มีมติ ให้ปรับแผน งบประมาณ สำหรับโครงการใหม่


## Inspect Results

The answer should mention the budget decision and avoid unsupported details.

In [6]:
summary_checks = [
    {"check": "mentions budget", "passed": "งบประมาณ" in answer},
    {"check": "mentions decision", "passed": "มติ" in answer or "ปรับแผน" in answer},
    {"check": "does not invent amount", "passed": "บาท" not in answer and "ล้าน" not in answer},
    {
        "check": "returns no evidence when context is empty",
        "passed": summarize_from_context("งบประมาณ", []) == "ไม่พบข้อมูลที่เกี่ยวข้องใน transcript",
    },
]

summary_checks_df = pd.DataFrame(summary_checks)
print(f"Passed {summary_checks_df['passed'].sum()}/{len(summary_checks_df)} summary checks")
summary_checks_df

Passed 4/4 summary checks


,check,passed
0,mentions budget,True
1,mentions decision,True
2,does not invent amount,True
3,returns no evidence when context is empty,True


In [7]:
assert summary_checks_df["passed"].all(), "Some query-based summarization checks failed"

summary_report = pd.DataFrame([
    {
        "question": question,
        "retrieved_chunks": len(retrieval_results),
        "answer": answer,
    }
])

summary_report

,question,retrieved_chunks,answer
0,ที่ประชุมมีมติเรื่องงบประมาณอย่างไร,1,จาก transcript ที่เกี่ยวข้อง: ที่ประชุม มีมติ ...


## What We Learned

- Query-based summarization answers one question, not the whole meeting.
- The answer must stay faithful to retrieved transcript context.
- A conservative extractive baseline is useful before adding a generative model.

## Common Problems

- Bad retrieval creates bad summaries.
- A generated summary can hallucinate details not present in the transcript.
- If ASR text is noisy, the answer may still be noisy.

## Optional Save

`outputs/` is ignored, so generated summaries stay local by default.

In [8]:
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output = {
        "question": question,
        "retrieved_context": retrieved_context,
        "answer": answer,
    }
    (OUTPUT_DIR / "query_summary.json").write_text(
        json.dumps(output, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Saved query summary to:", OUTPUT_DIR)

## Next Step

Build Phase 10: a small end-to-end demo that connects ASR, cleaning, retrieval, and query-based summarization.